<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

Система управления задачами внутри проекта «TaskManager»


### Вариант задания

Вариант № 23: базовый класс `Task` (задачи внутри проекта) и производные классы `DelegateTask`, `TeamTask`, `ResearchTask`.

<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Проект представляет собой консольную систему управления задачами внутри проекта. Базовый класс `Task` хранит общие данные задачи (идентификатор, название, приоритет, исполнитель, статус) и определяет методы `MarkAsComplete()`, `GetTaskDetails()` и `ReassignTo()`.

От него наследуются три класса:
- `DelegateTask` — делегированная задача с датой выполнения (`DueDate`); переопределяет `MarkAsComplete()`.
- `TeamTask` — командная задача с названием команды (`TeamName`); переопределяет `ReassignTo()`.
- `ResearchTask` — исследовательская задача с источником данных (`DataSource`); переопределяет `GetTaskDetails()`.

В проекте продемонстрированы: **инкапсуляция** (свойства с проверкой значений), **наследование**, **полиморфизм** (переопределение методов), **абстракция** (интерфейсы), **явная реализация интерфейса** (`ITaskReport`) и **управление зависимостями** (`TaskService` получает `ITaskRepository` и `INotifier` через конструктор).

#### Дополнительное задание
К базовому классу добавлены атрибуты `Description`, `Assignee`, `CreatedDate`, `IsCompleted` и методы `ChangePriority()`, `IsHighPriority()`, `GetStatus()`. В производные классы добавлены собственные атрибуты и методы (`IsOverdue()`, `AddMember()` и др.). Реализованы явная реализация интерфейса и внедрение зависимостей.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// ===== Интерфейсы =====
public interface ITaskReport
{
    string GenerateReport();
}

public interface INotifier
{
    void Notify(string message);
}

public interface ITaskRepository
{
    void Add(Task task);
    Task GetById(int taskId);
    IEnumerable<Task> GetAll();
}

// ===== Базовый класс =====
public class Task : ITaskReport
{
    private string _taskName;
    private int _priority;

    public int TaskId { get; }

    public string TaskName
    {
        get { return _taskName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название задачи не может быть пустым.");
            _taskName = value;
        }
    }

    public int Priority
    {
        get { return _priority; }
        set
        {
            if (value < 1 || value > 5)
                throw new ArgumentException("Приоритет должен быть от 1 до 5.");
            _priority = value;
        }
    }

    // Новые атрибуты
    public string Description { get; set; }
    public string Assignee { get; protected set; }
    public DateTime CreatedDate { get; }
    public bool IsCompleted { get; protected set; }

    public Task(int taskId, string taskName, int priority, string description, string assignee)
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
        Description = description;
        Assignee = assignee;
        CreatedDate = DateTime.Now;
        IsCompleted = false;
    }

    // Основные методы
    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        Console.WriteLine($"Задача «{TaskName}» выполнена.");
    }

    public virtual string GetTaskDetails()
    {
        return $"Задача #{TaskId}: {TaskName} | Приоритет: {Priority} | " +
               $"Исполнитель: {Assignee} | Статус: {GetStatus()}";
    }

    public virtual void ReassignTo(string newAssignee)
    {
        if (string.IsNullOrWhiteSpace(newAssignee))
            throw new ArgumentException("Имя исполнителя не может быть пустым.");
        Assignee = newAssignee;
        Console.WriteLine($"Задача «{TaskName}» переназначена: {Assignee}.");
    }

    // Новые методы
    public void ChangePriority(int newPriority)
    {
        Priority = newPriority;
        Console.WriteLine($"Приоритет задачи «{TaskName}» изменён на {Priority}.");
    }

    public bool IsHighPriority()
    {
        return Priority >= 4;
    }

    public string GetStatus()
    {
        return IsCompleted ? "Выполнена" : "В работе";
    }

    protected virtual string BuildReport()
    {
        return $"[ОТЧЁТ] #{TaskId} {TaskName}, приоритет {Priority}, статус: {GetStatus()}";
    }

    // Явная реализация интерфейса
    string ITaskReport.GenerateReport()
    {
        return BuildReport();
    }
}

// ===== Производный класс 1: делегированная задача =====
public class DelegateTask : Task
{
    public DateTime DueDate { get; set; }
    public string DelegatedBy { get; set; }

    public DelegateTask(int taskId, string taskName, int priority, string description,
                        string assignee, DateTime dueDate, string delegatedBy)
        : base(taskId, taskName, priority, description, assignee)
    {
        DueDate = dueDate;
        DelegatedBy = delegatedBy;
    }

    public override void MarkAsComplete()
    {
        base.MarkAsComplete();
        Console.WriteLine($"Срок выполнения: {DueDate:dd.MM.yyyy}. Задачу делегировал: {DelegatedBy}.");
    }

    public bool IsOverdue()
    {
        return !IsCompleted && DateTime.Now > DueDate;
    }

    protected override string BuildReport()
    {
        return base.BuildReport() + $", срок: {DueDate:dd.MM.yyyy}, просрочена: {(IsOverdue() ? "да" : "нет")}";
    }
}

// ===== Производный класс 2: командная задача =====
public class TeamTask : Task
{
    public string TeamName { get; set; }
    public List<string> Members { get; } = new List<string>();

    public TeamTask(int taskId, string taskName, int priority, string description,
                    string assignee, string teamName, params string[] members)
        : base(taskId, taskName, priority, description, assignee)
    {
        TeamName = teamName;
        Members.AddRange(members);
    }

    public void AddMember(string member)
    {
        if (!Members.Contains(member))
        {
            Members.Add(member);
            Console.WriteLine($"{member} добавлен в команду «{TeamName}».");
        }
    }

    public override void ReassignTo(string newAssignee)
    {
        if (!Members.Contains(newAssignee))
        {
            Console.WriteLine($"{newAssignee} не входит в команду «{TeamName}». Переназначение отменено.");
            return;
        }
        base.ReassignTo(newAssignee);
        Console.WriteLine($"Новый член команды «{TeamName}», ответственный за задачу: {newAssignee}.");
    }

    protected override string BuildReport()
    {
        return base.BuildReport() + $", команда: {TeamName} ({Members.Count} чел.)";
    }
}

// ===== Производный класс 3: исследовательская задача =====
public class ResearchTask : Task
{
    public string DataSource { get; set; }
    public int SourcesCount { get; set; }

    public ResearchTask(int taskId, string taskName, int priority, string description,
                        string assignee, string dataSource, int sourcesCount)
        : base(taskId, taskName, priority, description, assignee)
    {
        DataSource = dataSource;
        SourcesCount = sourcesCount;
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() + $" | Источник данных: {DataSource} (источников: {SourcesCount})";
    }

    public void AddSource()
    {
        SourcesCount++;
        Console.WriteLine($"Добавлен источник данных. Всего: {SourcesCount}.");
    }
}

// ===== Зависимости =====
public class InMemoryTaskRepository : ITaskRepository
{
    private readonly Dictionary<int, Task> _tasks = new Dictionary<int, Task>();

    public void Add(Task task)
    {
        _tasks[task.TaskId] = task;
    }

    public Task GetById(int taskId)
    {
        return _tasks.TryGetValue(taskId, out Task task) ? task : null;
    }

    public IEnumerable<Task> GetAll()
    {
        return _tasks.Values;
    }
}

public class ConsoleNotifier : INotifier
{
    public void Notify(string message)
    {
        Console.WriteLine($"[Консоль] {message}");
    }
}

public class EmailNotifier : INotifier
{
    public void Notify(string message)
    {
        Console.WriteLine($"[Email] {message}");
    }
}

// ===== Сервис: внедрение зависимостей через конструктор =====
public class TaskService
{
    private readonly ITaskRepository _repository;
    private readonly INotifier _notifier;

    public TaskService(ITaskRepository repository, INotifier notifier)
    {
        _repository = repository ?? throw new ArgumentNullException(nameof(repository));
        _notifier = notifier ?? throw new ArgumentNullException(nameof(notifier));
    }

    public void CreateTask(Task task)
    {
        _repository.Add(task);
        _notifier.Notify($"Создана задача: {task.TaskName} (приоритет {task.Priority}).");
    }

    public void CompleteTask(int taskId)
    {
        Task task = _repository.GetById(taskId);
        if (task == null)
        {
            _notifier.Notify($"Задача #{taskId} не найдена.");
            return;
        }
        task.MarkAsComplete();
        _notifier.Notify($"Задача #{taskId} завершена.");
    }

    public void ReassignTask(int taskId, string newAssignee)
    {
        Task task = _repository.GetById(taskId);
        if (task == null)
        {
            _notifier.Notify($"Задача #{taskId} не найдена.");
            return;
        }
        task.ReassignTo(newAssignee);
        _notifier.Notify($"Задача #{taskId}: исполнитель — {task.Assignee}.");
    }

    public void PrintReports()
    {
        foreach (Task task in _repository.GetAll())
        {
            ITaskReport report = task;
            Console.WriteLine(report.GenerateReport());
        }
    }
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
DelegateTask t1 = new DelegateTask(1, "Подготовить отчёт", 4, "Квартальный отчёт",
                                   "Али", new DateTime(2026, 11, 20), "Менеджер Карим");
DelegateTask t2 = new DelegateTask(2, "Обновить договор", 3, "Старый договор",
                                   "Сара", new DateTime(2026, 11, 1), "Директор");
TeamTask t3 = new TeamTask(3, "Разработать модуль", 5, "Модуль авторизации",
                           "Али", "Backend", "Али", "Сара", "Омид");
ResearchTask t4 = new ResearchTask(4, "Анализ рынка", 2, "Изучение конкурентов",
                                   "Омид", "Опросы и открытые отчёты", 3);

TaskService service = new TaskService(new InMemoryTaskRepository(), new ConsoleNotifier());

Console.WriteLine("--- Создание задач ---");
service.CreateTask(t1);
service.CreateTask(t2);
service.CreateTask(t3);
service.CreateTask(t4);

Console.WriteLine("\n--- GetTaskDetails() (полиморфизм) ---");
List<Task> tasks = new List<Task> { t1, t2, t3, t4 };
foreach (Task t in tasks)
    Console.WriteLine(t.GetTaskDetails());

Console.WriteLine("\n--- MarkAsComplete() ---");
service.CompleteTask(1);

Console.WriteLine("\n--- ReassignTo() ---");
service.ReassignTask(3, "Сара");
service.ReassignTask(3, "Рахим");

Console.WriteLine("\n--- Новые методы ---");
t4.ChangePriority(5);
Console.WriteLine($"Высокий приоритет: {t4.IsHighPriority()}");
Console.WriteLine($"t2 просрочена: {t2.IsOverdue()}");
t3.AddMember("Лейла");
t4.AddSource();

Console.WriteLine("\n--- Отчёты (явная реализация ITaskReport) ---");
service.PrintReports();

Console.WriteLine("\n--- Другой INotifier ---");
TaskService emailService = new TaskService(new InMemoryTaskRepository(), new EmailNotifier());
emailService.CreateTask(new ResearchTask(5, "Тест email", 1, "Проверка", "Али", "Данные", 1));

Console.WriteLine("\n--- Проверка валидации ---");
try { t1.Priority = 10; }
catch (ArgumentException ex) { Console.WriteLine($"Ошибка: {ex.Message}"); }

--- Создание задач ---
[Консоль] Создана задача: Подготовить отчёт (приоритет 4).
[Консоль] Создана задача: Обновить договор (приоритет 3).
[Консоль] Создана задача: Разработать модуль (приоритет 5).
[Консоль] Создана задача: Анализ рынка (приоритет 2).

--- GetTaskDetails() (полиморфизм) ---
Задача #1: Подготовить отчёт | Приоритет: 4 | Исполнитель: Али | Статус: В работе
Задача #2: Обновить договор | Приоритет: 3 | Исполнитель: Сара | Статус: В работе
Задача #3: Разработать модуль | Приоритет: 5 | Исполнитель: Али | Статус: В работе
Задача #4: Анализ рынка | Приоритет: 2 | Исполнитель: Омид | Статус: В работе | Источник данных: Опросы и открытые отчёты (источников: 3)

--- MarkAsComplete() ---
Задача «Подготовить отчёт» выполнена.
Срок выполнения: 20.11.2026. Задачу делегировал: Менеджер Карим.
[Консоль] Задача #1 завершена.

--- ReassignTo() ---
Задача «Разработать модуль» переназначена: Сара.
Новый член команды «Backend», ответственный за задачу: Сара.
[Консоль] Задача #3: исполнит